# Toxic Comment Detection — Brazilian Portuguese
**Dataset:** `comentarios_toxicos_ptBR.csv`  
**Model:** Logistic Regression + TF-IDF  
**Language:** Brazilian Portuguese 🇧🇷  

---
## Table of Contents
1. [Setup & Imports](#setup)
2. [Data Loading & Inspection](#data-loading)
3. [Exploratory Data Analysis (EDA)](#eda)
4. [Text Preprocessing & Cleaning](#preprocessing)
5. [Feature Engineering (TF-IDF)](#features)
6. [Model Training](#training)
7. [Evaluation](#evaluation)
8. [Inference (Single Prediction)](#inference)
9. [Save Artifacts](#save)

## 1. Setup & Imports <a id='setup'></a>

In [ ]:
import os
import re
import json
import string
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
print('All libraries imported successfully.')

## 2. Data Loading & Inspection <a id='data-loading'></a>

In [ ]:
# Adjust path if running notebook from a different directory
DATA_PATH = '../data/comentarios_toxicos_ptBR.csv'

df = pd.read_csv(DATA_PATH, index_col=0, encoding='utf-8')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())
print('\nClass distribution:')
print(df['toxic'].value_counts())

## 3. Exploratory Data Analysis (EDA) <a id='eda'></a>

In [ ]:
# Class distribution
counts = df['toxic'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Non-Toxic (0)', 'Toxic (1)'], counts.values,
              color=['#3b82d4', '#e74c3c'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}', ha='center', fontweight='bold')
ax.set_title('Class Distribution — Toxic vs Non-Toxic', fontweight='bold')
ax.set_ylabel('Number of Comments')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Comment length distribution
df['text_len'] = df['text'].str.len()
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df[df['toxic']==0]['text_len'], bins=60, alpha=0.7, label='Non-Toxic', color='#3b82d4')
ax.hist(df[df['toxic']==1]['text_len'], bins=60, alpha=0.7, label='Toxic',     color='#e74c3c')
ax.set_xlim(0, 500)
ax.set_title('Comment Length Distribution', fontweight='bold')
ax.set_xlabel('Character Count')
ax.set_ylabel('Frequency')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('Average comment length:')
print(df.groupby('toxic')['text_len'].describe().round(1))

## 4. Text Preprocessing & Cleaning <a id='preprocessing'></a>

In [ ]:
PT_STOPWORDS = {
    'a','ao','aos','aquela','aquelas','aquele','aqueles','aquilo','as','ate',
    'com','como','da','das','de','dela','delas','dele','deles','depois','do',
    'dos','e','ela','elas','ele','eles','em','entre','era','essa','essas',
    'esse','esses','esta','estas','este','estes','eu','foi','foram','ha',
    'isso','isto','ja','lhe','lhes','lo','mais','mas','me','mesmo','meu',
    'meus','minha','minhas','muito','na','nas','nao','nem','no','nos',
    'nossa','nossas','nosso','nossos','num','numa','o','os','ou','para',
    'pela','pelas','pelo','pelos','por','pra','qual','quando','que','quem',
    'se','seja','sem','seu','seus','si','so','sua','suas','tambem','te',
    'tem','ter','teu','teus','tua','tuas','um','uma','umas','uns','voce',
    'voces','vos','user','https','http','rt','co','via','vc','tb','td','ne',
}

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#\w+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [t for t in text.split() if t not in PT_STOPWORDS and len(t) > 2]
    return ' '.join(tokens)

# Drop nulls and duplicates
before = len(df)
df.dropna(subset=['text'], inplace=True)
df.drop_duplicates(subset=['text'], inplace=True)
print(f'Removed {before - len(df)} null/duplicate rows. Remaining: {len(df):,}')

df['clean_text'] = df['text'].apply(clean_text)
print('Sample cleaned text:')
df[['text','clean_text','toxic']].head(5)

In [ ]:
# Top 20 words in toxic comments
toxic_words = ' '.join(df[df['toxic']==1]['clean_text'].values)
word_counts = Counter(toxic_words.split()).most_common(20)
words, freqs = zip(*word_counts)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(list(words)[::-1], list(freqs)[::-1], color='#e74c3c', edgecolor='white')
ax.set_title('Top 20 Most Frequent Words in Toxic Comments', fontweight='bold')
ax.set_xlabel('Frequency')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Feature Engineering (TF-IDF) <a id='features'></a>

In [ ]:
X = df['clean_text']
y = df['toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print(f'Training set: {X_train_vec.shape}')
print(f'Test set:     {X_test_vec.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')

## 6. Model Training <a id='training'></a>

In [ ]:
model = LogisticRegression(
    C=1.0, max_iter=1000, solver='lbfgs',
    class_weight='balanced', random_state=42
)
model.fit(X_train_vec, y_train)
print('Model trained successfully.')

## 7. Evaluation <a id='evaluation'></a>

In [ ]:
y_pred = model.predict(X_test_vec)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-Toxic', 'Toxic']))

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-Score : {f1:.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Toxic','Toxic'],
            yticklabels=['Non-Toxic','Toxic'],
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metric_values = [acc, prec, rec, f1]
bars = axes[1].bar(metric_names, metric_values,
                   color=['#3b82d4','#7c5cd8','#e67e22','#e74c3c'], edgecolor='white')
for bar, val in zip(bars, metric_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_ylim(0, 1.0)
axes[1].set_title('Evaluation Metrics', fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 8. Inference (Single Prediction) <a id='inference'></a>

In [ ]:
def predict_comment(text):
    cleaned = clean_text(text)
    vec     = vectorizer.transform([cleaned])
    pred    = model.predict(vec)[0]
    proba   = model.predict_proba(vec)[0]
    label   = 'Toxic' if pred == 1 else 'Non-Toxic'
    print(f'Comment : {text}')
    print(f'Label   : {label}')
    print(f'Toxic%  : {proba[1]:.2%}')
    print()

test_comments = [
    'Voce e um idiota completo, vai se ferrar!',
    'Bom dia! Espero que voce tenha um otimo dia.',
    'Esse time e uma merda, horrivel demais',
    'Que delicia de receita, vou fazer amanha!',
    'Cala a boca seu burro inutil',
]
for c in test_comments:
    predict_comment(c)

## 9. Save Artifacts <a id='save'></a>

In [ ]:
MODEL_DIR = '../model'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(model,      os.path.join(MODEL_DIR, 'toxic_model.pkl'))
joblib.dump(vectorizer, os.path.join(MODEL_DIR, 'vectorizer.pkl'))

metrics_out = {
    'accuracy':  round(float(acc),  4),
    'precision': round(float(prec), 4),
    'recall':    round(float(rec),  4),
    'f1_score':  round(float(f1),   4),
    'confusion_matrix': cm.tolist(),
    'train_samples': int(len(X_train)),
    'test_samples':  int(len(X_test)),
    'total_samples': int(len(df)),
}
with open(os.path.join(MODEL_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('Artifacts saved:')
print('  model/toxic_model.pkl')
print('  model/vectorizer.pkl')
print('  model/metrics.json')